# 🧪 Portfolio Backtesting System - System Test

**วัตถุประสงค์:** ทดสอบว่าระบบทำงานได้ถูกต้อง ไม่มี error

**ขั้นตอน:**
1. ทดสอบ Data Loading จาก MySQL
2. ทดสอบ Risk Metrics Calculation
3. ทดสอบ Portfolio Optimization
4. ทดสอบ Basic Backtesting
5. สร้าง Sample Charts

**ข้อมูล:** Weekly price history (41,800 rows, 50 ETFs, 16 years)

---

## Cell 1: Import Libraries & Setup

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import mysql.connector
from scipy.optimize import minimize
from typing import Dict, List, Optional, Union

# ==================== DATA LOADER ====================

class DataLoader:
    """Load data from MySQL database"""

    def __init__(self, config: Dict):
        self.config = config
        self.conn = None

    def connect(self):
        """Connect to MySQL database"""
        try:
            self.conn = mysql.connector.connect(**self.config)
            return True
        except mysql.connector.Error as e:
            print(f"❌ Connection error: {e}")
            return False

    def disconnect(self):
        """Close database connection"""
        if self.conn:
            self.conn.close()

    def get_etfs(self) -> pd.DataFrame:
        """Get all ETFs"""
        query = """
            SELECT etf_id, ticker_symbol, etf_name, asset_class,
                   expense_ratio, inception_date
            FROM etf_master
            ORDER BY ticker_symbol
        """
        return pd.read_sql(query, self.conn)

    def get_benchmarks(self) -> pd.DataFrame:
        """Get all benchmark portfolios"""
        query = """
            SELECT benchmark_id, benchmark_name, description, risk_level
            FROM benchmark_portfolios
            ORDER BY benchmark_name
        """
        return pd.read_sql(query, self.conn)

    def get_benchmark_holdings(self, benchmark_id: int) -> pd.DataFrame:
        """Get holdings for a specific benchmark"""
        query = """
            SELECT
                bh.benchmark_id,
                b.benchmark_name,
                e.ticker_symbol,
                e.etf_name,
                bh.target_weight
            FROM benchmark_holdings bh
            JOIN benchmark_portfolios b ON bh.benchmark_id = b.benchmark_id
            JOIN etf_master e ON bh.etf_id = e.etf_id
            WHERE bh.benchmark_id = %s
            ORDER BY bh.target_weight DESC
        """
        return pd.read_sql(query, self.conn, params=(benchmark_id,))

    def get_price_history(self, tickers: Optional[List[str]] = None,
                         start_date: Optional[str] = None,
                         end_date: Optional[str] = None) -> pd.DataFrame:
        """Get price history for ETFs"""
        query = """
            SELECT
                e.ticker_symbol,
                e.etf_id,
                ph.date,
                ph.open,
                ph.high,
                ph.low,
                ph.close,
                ph.adj_close,
                ph.volume
            FROM price_history ph
            JOIN etf_master e ON ph.etf_id = e.etf_id
            WHERE 1=1
        """

        params = []

        if tickers:
            placeholders = ','.join(['%s'] * len(tickers))
            query += f" AND e.ticker_symbol IN ({placeholders})"
            params.extend(tickers)

        if start_date:
            query += " AND ph.date >= %s"
            params.append(start_date)

        if end_date:
            query += " AND ph.date <= %s"
            params.append(end_date)

        query += " ORDER BY e.ticker_symbol, ph.date"

        if params:
            return pd.read_sql(query, self.conn, params=params)
        else:
            return pd.read_sql(query, self.conn)


# ==================== RISK METRICS ====================

class RiskMetrics:
    """Calculate risk metrics for portfolio analysis"""

    @staticmethod
    def sharpe_ratio(returns: Union[np.ndarray, pd.Series],
                     risk_free_rate: float = 0.02,
                     periods_per_year: int = 52) -> float:
        """Calculate Sharpe Ratio"""
        if len(returns) == 0:
            return 0.0
        excess_returns = returns - (risk_free_rate / periods_per_year)
        if excess_returns.std() == 0:
            return 0.0
        sharpe = (excess_returns.mean() / excess_returns.std()) * np.sqrt(periods_per_year)
        return float(sharpe)

    @staticmethod
    def sortino_ratio(returns: Union[np.ndarray, pd.Series],
                      risk_free_rate: float = 0.02,
                      periods_per_year: int = 52) -> float:
        """Calculate Sortino Ratio (uses downside deviation)"""
        if len(returns) == 0:
            return 0.0
        excess_returns = returns - (risk_free_rate / periods_per_year)
        downside_returns = excess_returns[excess_returns < 0]
        if len(downside_returns) == 0 or downside_returns.std() == 0:
            return 0.0
        sortino = (excess_returns.mean() / downside_returns.std()) * np.sqrt(periods_per_year)
        return float(sortino)

    @staticmethod
    def max_drawdown(returns: Union[np.ndarray, pd.Series]) -> float:
        """Calculate Maximum Drawdown"""
        if len(returns) == 0:
            return 0.0
        cumulative = (1 + returns).cumprod()
        running_max = np.maximum.accumulate(cumulative)
        drawdown = (cumulative - running_max) / running_max
        return float(drawdown.min())

    @staticmethod
    def volatility(returns: Union[np.ndarray, pd.Series],
                   periods_per_year: int = 52) -> float:
        """Calculate annualized volatility"""
        if len(returns) == 0:
            return 0.0
        return float(returns.std() * np.sqrt(periods_per_year))

    @staticmethod
    def value_at_risk(returns: Union[np.ndarray, pd.Series],
                      confidence_level: float = 0.95) -> float:
        """Calculate Value at Risk (VaR)"""
        if len(returns) == 0:
            return 0.0
        return float(np.percentile(returns, (1 - confidence_level) * 100))

    @staticmethod
    def calmar_ratio(returns: Union[np.ndarray, pd.Series],
                     periods_per_year: int = 52) -> float:
        """Calculate Calmar Ratio (Return / Max Drawdown)"""
        if len(returns) == 0:
            return 0.0
        annual_return = returns.mean() * periods_per_year
        max_dd = abs(RiskMetrics.max_drawdown(returns))
        if max_dd == 0:
            return 0.0
        return float(annual_return / max_dd)


# ==================== PORTFOLIO OPTIMIZER ====================

class PortfolioOptimizer:
    """Optimize portfolio weights using mean-variance optimization"""

    def __init__(self, returns: pd.DataFrame):
        self.returns = returns
        self.mean_returns = returns.mean()
        self.cov_matrix = returns.cov()
        self.n_assets = len(returns.columns)

    def optimize_sharpe(self, risk_free_rate: float = 0.02) -> Dict[str, float]:
        """Optimize for maximum Sharpe ratio"""
        def objective(weights):
            portfolio_return = np.dot(weights, self.mean_returns) * 52
            portfolio_std = np.sqrt(np.dot(weights.T, np.dot(self.cov_matrix * 52, weights)))
            if portfolio_std == 0:
                return 1e10
            sharpe = (portfolio_return - risk_free_rate) / portfolio_std
            return -sharpe

        constraints = [{'type': 'eq', 'fun': lambda x: np.sum(x) - 1}]
        bounds = tuple((0, 1) for _ in range(self.n_assets))
        x0 = np.array([1.0 / self.n_assets] * self.n_assets)

        result = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=constraints)

        if result.success:
            weights = dict(zip(self.returns.columns, result.x))
            weights = {k: round(v, 4) for k, v in weights.items()}
            weights = {k: v for k, v in weights.items() if v >= 0.0001}
            return weights
        else:
            return dict(zip(self.returns.columns, [1.0/self.n_assets] * self.n_assets))

    def optimize_min_volatility(self) -> Dict[str, float]:
        """Optimize for minimum volatility"""
        def objective(weights):
            return np.sqrt(np.dot(weights.T, np.dot(self.cov_matrix, weights)))

        constraints = [{'type': 'eq', 'fun': lambda x: np.sum(x) - 1}]
        bounds = tuple((0, 1) for _ in range(self.n_assets))
        x0 = np.array([1.0 / self.n_assets] * self.n_assets)

        result = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=constraints)

        if result.success:
            weights = dict(zip(self.returns.columns, result.x))
            weights = {k: round(v, 4) for k, v in weights.items()}
            weights = {k: v for k, v in weights.items() if v >= 0.0001}
            return weights
        else:
            return dict(zip(self.returns.columns, [1.0/self.n_assets] * self.n_assets))

    def calculate_portfolio_metrics(self, weights: Dict[str, float]) -> Dict[str, float]:
        """Calculate portfolio metrics for given weights"""
        w = np.array([weights.get(col, 0) for col in self.returns.columns])
        portfolio_return = np.dot(w, self.mean_returns) * 52
        portfolio_vol = np.sqrt(np.dot(w.T, np.dot(self.cov_matrix * 52, w)))
        sharpe = (portfolio_return - 0.02) / portfolio_vol if portfolio_vol > 0 else 0

        return {
            'annual_return': portfolio_return,
            'annual_volatility': portfolio_vol,
            'sharpe_ratio': sharpe
        }

# Plot settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# MySQL Configuration
MYSQL_CONFIG = {
    'host': '127.0.0.1',
    'port': 3306,
    'user': 'root',
    'password': 'krittanut123456',
    'database': 'portfolio_backtesting'
}

print("✅ Classes loaded: DataLoader, RiskMetrics, PortfolioOptimizer")
print(f"📊 Database: {MYSQL_CONFIG['database']} @ {MYSQL_CONFIG['host']}:{MYSQL_CONFIG['port']}")

---

## ✅ Test 1: Data Loading (5 นาที)

ทดสอบว่าโหลดข้อมูลจาก MySQL ได้หรือไม่

In [ ]:
print("="*70)
print("Test 1: Data Loading from MySQL".center(70))
print("="*70)
print()

# Initialize DataLoader
loader = DataLoader(MYSQL_CONFIG)

# Connect
if loader.connect():
    print("✅ Connected to MySQL")
else:
    print("❌ Connection failed!")
    raise Exception("Cannot connect to database")

# Load ETFs
print("\n📊 Loading ETFs...")
etfs = loader.get_etfs()
print(f"   ✅ Loaded {len(etfs)} ETFs")
print(f"\n{etfs.head(10)}")

# Load Benchmarks
print("\n📊 Loading Benchmarks...")
benchmarks = loader.get_benchmarks()
print(f"   ✅ Loaded {len(benchmarks)} Benchmarks")
print(f"\n{benchmarks.head(10)}")

# Load Price History
print("\n📊 Loading Price History...")
prices = loader.get_price_history()
print(f"   ✅ Loaded {len(prices):,} price records")
print(f"   📅 Date range: {prices['date'].min()} to {prices['date'].max()}")
print(f"   📊 Tickers: {prices['ticker_symbol'].nunique()}")
print(f"\n{prices.head(10)}")

print("\n" + "="*70)
print("✅ Test 1 PASSED - Data loading successful!".center(70))
print("="*70)

---

## ✅ Test 2: Calculate Returns (5 นาที)

แปลง price data เป็น returns

In [ ]:
print("="*70)
print("Test 2: Calculate Returns".center(70))
print("="*70)
print()

# Pivot price data
print("📊 Pivoting price data...")
price_pivot = prices.pivot(index='date', columns='ticker_symbol', values='adj_close')
price_pivot = price_pivot.sort_index()
print(f"   ✅ Price matrix: {price_pivot.shape}")
print(f"   📅 Dates: {len(price_pivot)} weeks")
print(f"   📊 Assets: {len(price_pivot.columns)} ETFs")

# Calculate returns
print("\n📈 Calculating returns...")
returns = price_pivot.pct_change().dropna()
print(f"   ✅ Returns matrix: {returns.shape}")

# Summary statistics
print("\n📊 Returns Summary:")
print(returns.describe())

print("\n" + "="*70)
print("✅ Test 2 PASSED - Returns calculated!".center(70))
print("="*70)

---

## ✅ Test 3: Risk Metrics (5 นาที)

ทดสอบการคำนวณ risk metrics

In [ ]:
print("="*70)
print("Test 3: Risk Metrics Calculation".center(70))
print("="*70)
print()

# Test with SPY, AGG, GLD
test_tickers = ['SPY', 'AGG', 'GLD']

risk_results = []

for ticker in test_tickers:
    if ticker in returns.columns:
        ticker_returns = returns[ticker].values
        
        sharpe = RiskMetrics.sharpe_ratio(ticker_returns)
        sortino = RiskMetrics.sortino_ratio(ticker_returns)
        max_dd = RiskMetrics.max_drawdown(ticker_returns)
        volatility = RiskMetrics.volatility(ticker_returns)
        
        risk_results.append({
            'Ticker': ticker,
            'Sharpe Ratio': sharpe,
            'Sortino Ratio': sortino,
            'Max Drawdown': max_dd,
            'Annual Volatility': volatility
        })
        
        print(f"\n{ticker}:")
        print(f"  Sharpe Ratio:      {sharpe:7.2f}")
        print(f"  Sortino Ratio:     {sortino:7.2f}")
        print(f"  Max Drawdown:      {max_dd:7.2%}")
        print(f"  Annual Volatility: {volatility:7.2%}")

# Create DataFrame
risk_df = pd.DataFrame(risk_results)

print("\n" + "="*70)
print("📊 Risk Metrics Summary:")
print("="*70)
print(risk_df.to_string(index=False))

print("\n" + "="*70)
print("✅ Test 3 PASSED - Risk metrics calculated!".center(70))
print("="*70)

---

## ✅ Test 4: Portfolio Optimization (10 นาที)

ทดสอบ portfolio optimization

In [ ]:
print("="*70)
print("Test 4: Portfolio Optimization".center(70))
print("="*70)
print()

# Use 5 ETFs for testing
test_assets = ['SPY', 'AGG', 'GLD', 'VTI', 'BND']
test_returns = returns[test_assets]

print(f"📊 Testing with {len(test_assets)} assets: {', '.join(test_assets)}")
print()

# Initialize optimizer
optimizer = PortfolioOptimizer(test_returns)

# Optimize for maximum Sharpe ratio
print("🎯 Optimizing for Maximum Sharpe Ratio...")
optimal_weights = optimizer.optimize_sharpe()

print("\n✅ Optimal Weights:")
for ticker, weight in optimal_weights.items():
    print(f"  {ticker}: {weight:7.2%}")

total_weight = sum(optimal_weights.values())
print(f"\n  Total: {total_weight:7.2%} (should be ~100%)")

# Calculate portfolio metrics
metrics = optimizer.calculate_portfolio_metrics(optimal_weights)

print("\n📊 Optimized Portfolio Metrics:")
print(f"  Annual Return:     {metrics['annual_return']:7.2%}")
print(f"  Annual Volatility: {metrics['annual_volatility']:7.2%}")
print(f"  Sharpe Ratio:      {metrics['sharpe_ratio']:7.2f}")

print("\n" + "="*70)
print("✅ Test 4 PASSED - Portfolio optimization working!".center(70))
print("="*70)

---

## ✅ Test 5: Visualization (10 นาที)

ทดสอบการสร้าง charts พื้นฐาน

In [ ]:
print("="*70)
print("Test 5: Basic Charts".center(70))
print("="*70)
print()

# Chart 1: Price Performance (Normalized to 100)
print("📈 Chart 1: Price Performance (Normalized)")

normalized = (price_pivot / price_pivot.iloc[0]) * 100

fig, ax = plt.subplots(figsize=(12, 6))

for ticker in ['SPY', 'AGG', 'GLD', 'BND', 'VTI']:
    if ticker in normalized.columns:
        ax.plot(normalized.index, normalized[ticker], label=ticker, linewidth=2)

ax.set_title('ETF Performance (Base 100)', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Value (Base 100)', fontsize=12)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('charts/test_price_performance.png', dpi=300, bbox_inches='tight')
print("   ✅ Saved: charts/test_price_performance.png")
plt.show()

print("\n" + "="*70)
print("✅ Test 5 PASSED - Charts created!".center(70))
print("="*70)

In [ ]:
# Chart 2: Correlation Heatmap
print("📊 Chart 2: Correlation Heatmap")

# Use subset of ETFs for clarity
subset_tickers = ['SPY', 'AGG', 'GLD', 'BND', 'VTI', 'QQQ', 'IWM', 'TLT', 'GLD', 'VNQ']
subset_returns = returns[[t for t in subset_tickers if t in returns.columns]]

corr_matrix = subset_returns.corr()

fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=1, cbar_kws={'shrink': 0.8})

ax.set_title('ETF Return Correlation Matrix', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig('charts/test_correlation_heatmap.png', dpi=300, bbox_inches='tight')
print("   ✅ Saved: charts/test_correlation_heatmap.png")
plt.show()

In [ ]:
# Chart 3: Risk-Return Scatter
print("📊 Chart 3: Risk-Return Scatter")

# Calculate annual metrics for all ETFs
annual_returns = returns.mean() * 52
annual_volatility = returns.std() * np.sqrt(52)

fig, ax = plt.subplots(figsize=(12, 7))

ax.scatter(annual_volatility, annual_returns, s=100, alpha=0.6, c='steelblue', edgecolors='black')

# Annotate selected ETFs
for ticker in ['SPY', 'AGG', 'GLD', 'BND', 'TLT']:
    if ticker in annual_returns.index:
        ax.annotate(ticker,
                   (annual_volatility[ticker], annual_returns[ticker]),
                   xytext=(10, 10), textcoords='offset points',
                   fontsize=12, fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.5))

ax.set_title('Risk-Return Profile (All ETFs)', fontsize=16, fontweight='bold')
ax.set_xlabel('Annual Volatility (Risk)', fontsize=12)
ax.set_ylabel('Annual Return', fontsize=12)
ax.grid(True, alpha=0.3)

# Format axes as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))

plt.tight_layout()
plt.savefig('charts/test_risk_return.png', dpi=300, bbox_inches='tight')
print("   ✅ Saved: charts/test_risk_return.png")
plt.show()

---

## 🎉 SUMMARY: All Tests Complete!

In [ ]:
print("="*70)
print("🎉 SYSTEM TEST SUMMARY".center(70))
print("="*70)
print()
print("✅ Test 1: Data Loading          PASSED")
print("✅ Test 2: Calculate Returns      PASSED")
print("✅ Test 3: Risk Metrics           PASSED")
print("✅ Test 4: Portfolio Optimization PASSED")
print("✅ Test 5: Visualization          PASSED")
print()
print("="*70)
print("📊 Data Summary:")
print("="*70)
print(f"  ETFs:              {len(etfs)}")
print(f"  Benchmarks:        {len(benchmarks)}")
print(f"  Price Records:     {len(prices):,}")
print(f"  Date Range:        {prices['date'].min()} to {prices['date'].max()}")
print(f"  Returns Matrix:    {returns.shape[0]} weeks × {returns.shape[1]} assets")
print()
print("="*70)
print("📁 Charts Created:")
print("="*70)
print("  ✅ charts/test_price_performance.png")
print("  ✅ charts/test_correlation_heatmap.png")
print("  ✅ charts/test_risk_return.png")
print()
print("="*70)
print("🎉 ALL SYSTEMS OPERATIONAL!".center(70))
print("="*70)
print()
print("💡 Next Steps:")
print("  1. Run complete backtesting (main.ipynb)")
print("  2. Create full analysis (analytics.ipynb)")
print("  3. Write final report")
print()
print("✅ System is ready for production analysis!")
print()

# Cleanup
loader.disconnect()
print("✅ Database connection closed")